# 08 - Statistical Analysis and Inflation Effects

**Purpose:** Quantify inflation (% and absolute) vs P0, with moving-block bootstrap CIs, paired tests with correction, Friedman+Nemenyi, and effect sizes.

**Inputs:** results/metrics/ (P0 and P1-P6 results from notebooks 06-07).

**Outputs:** results/tables/ inflation tables; results/figures/ inflation figures.

**Runtime estimate:** 10-30 minutes.

## Compute inflation vs P0 baseline

In [1]:
import pandas as pd

from sdg_leakage_audit.evaluation.inflation import compute_inflation
from sdg_leakage_audit.utils.paths import results_figures_dir, results_metrics_dir, results_tables_dir, to_project_relative

p0_df = pd.read_csv(results_metrics_dir() / "p0_baseline_metrics.csv")
p1_p6_df = pd.read_csv(results_metrics_dir() / "p1_p6_leakage_metrics.csv")
all_results_df = pd.concat([p0_df, p1_p6_df], ignore_index=True)

inflation_df = compute_inflation(all_results_df, baseline_protocol="P0")

out_path = results_tables_dir() / "inflation_by_cell.csv"
inflation_df.to_csv(out_path, index=False)
print(f"Saved per-cell inflation table ({len(inflation_df)} rows) -> {to_project_relative(out_path)}")

inflation_summary = (
    inflation_df.groupby("protocol_id")[["mae_inflation_pct", "skill_score_inflation_abs"]]
    .mean()
    .round(2)
)
inflation_summary


Saved per-cell inflation table (840 rows) -> results\tables\inflation_by_cell.csv


,mae_inflation_pct,skill_score_inflation_abs
protocol_id,,
P0,0.00,0.00
P1,0.85,0.15
P2,-1.43,0.02
P3,1.32,0.02
P4,2.17,0.07
P5,1.65,0.04
P6,3.61,0.11


## Moving-block bootstrap confidence intervals

In [2]:
from sdg_leakage_audit.stats.bootstrap import moving_block_bootstrap_ci
from sdg_leakage_audit.utils.io import read_yaml
from sdg_leakage_audit.utils.paths import configs_dir

experiment_cfg = read_yaml(configs_dir() / "experiment.yaml")["experiment"]
bootstrap_cfg = experiment_cfg["bootstrap"]
N_RESAMPLES = 500  # reduced from the full profile's 2000 for fast-profile runtime
CONFIDENCE_LEVEL = bootstrap_cfg["confidence_level"]

bootstrap_rows = []
for protocol_id, group in inflation_df[inflation_df["protocol_id"] != "P0"].groupby("protocol_id"):
    values = group["mae_inflation_pct"].dropna().to_numpy()
    if len(values) < 2:
        continue
    # block_length heuristic: min(len(values), max(2, number of distinct horizons)),
    # since consecutive rows for the same (dataset, model) across seeds are only weakly
    # correlated at this grid granularity -- a short block is appropriate here.
    block_length = min(len(values), 3)
    point, lower, upper = moving_block_bootstrap_ci(
        values, block_length=block_length, n_resamples=N_RESAMPLES, confidence_level=CONFIDENCE_LEVEL, seed=0
    )
    bootstrap_rows.append(
        {"protocol_id": protocol_id, "mean_mae_inflation_pct": point, "ci_lower": lower, "ci_upper": upper, "n": len(values)}
    )

bootstrap_df = pd.DataFrame(bootstrap_rows).set_index("protocol_id").round(2)
bootstrap_df


,mean_mae_inflation_pct,ci_lower,ci_upper,n
protocol_id,,,,
P1,0.85,-13.81,16.21,120
P2,-1.43,-3.74,1.01,120
P3,1.32,-0.60,3.37,120
P4,2.17,-2.05,6.79,120
P5,1.65,-1.30,4.04,120
P6,3.61,-11.35,18.37,120


## Paired significance tests with multiple-comparison correction

In [3]:
from sdg_leakage_audit.stats.tests import paired_tests_with_correction

paired_results = paired_tests_with_correction(all_results_df, metric="mae", method="holm")
out_path = results_tables_dir() / "paired_tests_mae.csv"
paired_results.to_csv(out_path, index=False)
print(f"Saved {len(paired_results)} pairwise comparisons -> {to_project_relative(out_path)}")
paired_results[paired_results["significant"]].sort_values("pvalue_corrected")


Saved 21 pairwise comparisons -> results\tables\paired_tests_mae.csv


,protocol_a,protocol_b,n_pairs,statistic,pvalue,pvalue_corrected,significant
10,P1,P6,120,1810.0,0.000002,0.000039,True
14,P2,P6,120,2304.0,0.000515,0.010308,True
5,P0,P6,120,2314.0,0.000568,0.010793,True
15,P3,P4,120,2354.0,0.000833,0.014989,True
17,P3,P6,120,2359.0,0.000873,0.014989,True


## Friedman omnibus test and Nemenyi post-hoc

In [4]:
from sdg_leakage_audit.stats.tests import friedman_nemenyi

friedman_result = friedman_nemenyi(all_results_df, metric="mae")
print(
    f"Friedman chi-square = {friedman_result['friedman_statistic']:.3f}, "
    f"p = {friedman_result['friedman_pvalue']:.4g} "
    f"(significant difference among protocols: {friedman_result['friedman_pvalue'] < 0.05})"
)

out_path = results_tables_dir() / "nemenyi_pvalues_mae.csv"
friedman_result["nemenyi_pvalues"].to_csv(out_path)
print(f"Saved Nemenyi pairwise p-value matrix -> {to_project_relative(out_path)}")
friedman_result["nemenyi_pvalues"].round(4)


Friedman chi-square = 32.752, p = 1.17e-05 (significant difference among protocols: True)
Saved Nemenyi pairwise p-value matrix -> results\tables\nemenyi_pvalues_mae.csv


,P0,P1,P2,P3,P4,P5,P6
P0,1.0000,0.2024,0.9998,0.9948,0.8846,0.8723,0.0020
P1,0.2024,1.0000,0.0865,0.0373,0.9071,0.9173,0.7486
P2,0.9998,0.0865,1.0000,0.9999,0.6929,0.6736,0.0004
P3,0.9948,0.0373,0.9999,1.0000,0.4920,0.4720,0.0001
P4,0.8846,0.9071,0.6929,0.4920,1.0000,1.0000,0.1088
P5,0.8723,0.9173,0.6736,0.4720,1.0000,1.0000,0.1171
P6,0.0020,0.7486,0.0004,0.0001,0.1088,0.1171,1.0000


## Effect sizes

In [5]:
from sdg_leakage_audit.stats.tests import effect_size_cliffs_delta

effect_size_rows = []
p0_mae = all_results_df.loc[all_results_df["protocol_id"] == "P0", "mae"].to_numpy()
for protocol_id in ["P1", "P2", "P3", "P4", "P5", "P6"]:
    protocol_mae = all_results_df.loc[all_results_df["protocol_id"] == protocol_id, "mae"].to_numpy()
    delta = effect_size_cliffs_delta(protocol_mae, p0_mae)
    effect_size_rows.append({"protocol_id": protocol_id, "cliffs_delta_vs_p0": round(delta, 3)})

effect_size_df = pd.DataFrame(effect_size_rows).set_index("protocol_id")
effect_size_df


,cliffs_delta_vs_p0
protocol_id,
P1,-0.076
P2,0.003
P3,-0.011
P4,-0.023
P5,-0.011
P6,-0.086


In [6]:
from sdg_leakage_audit.viz.plots import plot_inflation_bars
from sdg_leakage_audit.viz.style import apply_plot_style

apply_plot_style()
plot_inflation_bars(inflation_df, metric="mae", save_path=results_figures_dir() / "08_inflation_bars_mae.png")
plot_inflation_bars(inflation_df, metric="skill_score", save_path=results_figures_dir() / "08_inflation_bars_skill_score.png")
print("Saved inflation bar charts to results/figures/.")


2026-10-21 19:09:13 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\08_inflation_bars_mae.png


2026-10-21 19:09:14 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\08_inflation_bars_skill_score.png


Saved inflation bar charts to results/figures/.


## Findings / what to carry forward

- Per-cell inflation (both `_abs` and `_pct`, signed so positive always means "this protocol's
  error looks more flattering than it should") saved to `results/tables/inflation_by_cell.csv`;
  mean inflation by protocol summarized in this notebook's first output table.
- Moving-block bootstrap 95% CIs on mean MAE-inflation-% per protocol are in the bootstrap
  table above; any protocol whose CI excludes 0 has a directionally reliable (not just
  noise-level) inflation effect at this sample size.
- Paired Wilcoxon tests (Holm-corrected) between every pair of protocols are saved to
  `results/tables/paired_tests_mae.csv`; only the significant pairs (after correction) are
  shown inline.
- Friedman omnibus test result and the full Nemenyi pairwise p-value matrix are saved to
  `results/tables/nemenyi_pvalues_mae.csv` -- this is the test that answers "do the 7 protocols
  differ at all" before drilling into which specific pairs differ.
- Cliff's delta effect sizes (each leaky protocol's MAE distribution vs P0's) give a
  scale-free, non-parametric measure of how large each protocol's shift is, independent of
  statistical significance (which is sensitive to this run's small `fast`-profile sample size).
- Caveat carried forward from notebook 07: P6's rows in `all_results_df` are CV-estimate
  metrics (averaged over 5 folds) being compared against every other protocol's single-split
  metrics in all of the tests above -- a known methodological mismatch, stated here again
  rather than silently absorbed into the numbers.
- Sample-size caveat: this entire notebook ran on the `fast` profile's single seed and small
  hyperparameter grids, so these significance tests have limited power; re-running under the
  `full` profile (multiple seeds) is necessary before treating any of these p-values as
  trustworthy evidence rather than a pipeline-correctness check.
- Next (notebook 09): translate the inflation magnitudes computed here into SDG-relevant
  decision consequences (reserve margin, exceedance alerts, storage sizing) per dataset.
